In [1]:
import random
import cmath
import numpy as np

def generate_dataset(equations,fixed_msbs,modulus,secret_key):
    hj = []
    cj = []
    max_nonce = modulus >> fixed_msbs
    for _ in range(equations):
        # biased nonce
        k = random.randrange(max_nonce)
        # random multiplier
        c = random.randrange(modulus)
        # h = k - c*x mod n
        h = (k - c * secret_key) % modulus
        hj.append(h)
        cj.append(c)
    return hj, cj

def compute_bias(hj, cj, modulus, candidate):
    total = 0
    for h, c in zip(hj, cj):
        angle = 2j * cmath.pi * ((h + c * candidate) % modulus) / modulus
        total += cmath.exp(angle)
    return abs(total / len(hj))

def sort_and_reduce(hj, cj, modulus):
    pairs = sorted(zip(cj, hj))
    new_hj = []
    new_cj = []
    for i in range(len(pairs) - 1):
        c0, h0 = pairs[i]
        c1, h1 = pairs[i + 1]
        new_c = (c1 - c0) % modulus
        new_h = (h1 - h0) % modulus
        new_cj.append(new_c)
        new_hj.append(new_h)
    return new_hj, new_cj

def filter_pairs(hj, cj, ell):
    bound = 1 << ell
    filtered_h = []
    filtered_c = []
    for h, c in zip(hj, cj):
        if c < bound:
            filtered_h.append(h)
            filtered_c.append(c)
    return filtered_h, filtered_c
def build_Z(hj, cj, modulus, ell):
    size = 1 << ell
    Z = np.zeros(size, dtype=np.complex128)
    for h, c in zip(hj, cj):
        Z[c] += cmath.exp(2j * cmath.pi * h / modulus)
    return Z

def print_msbs(a, b, L):
    n = max(a.bit_length(), b.bit_length())
    print(bin(a >> (n - L))[2:])
    print(bin(b >> (n - L))[2:])

def run_ifft(Z):
    return np.fft.ifft(Z)


n = 13441
x = 11000
equations = 10000
fixed_msbs = 1
iterations = 4
ell = 8
hj, cj = generate_dataset(equations,fixed_msbs,n,x)
print("initial bias =", compute_bias(hj, cj, n, x))

for i in range(iterations):
    print()
    print("iteration", i)
    print("pairs =", len(cj))
    print("bias =", compute_bias(hj, cj, n, x))
    hj_small, cj_small = filter_pairs(hj, cj, ell)
    print("filtered pairs =", len(cj_small))
    if len(cj_small) == 0:
        continue
    # build Z vector
    Z = build_Z(hj_small, cj_small, n, ell)
    print("Z size =", len(Z))
    # inverse FFT
    W = run_ifft(Z)
    # strongest peak
    peak = np.argmax(np.abs(W))
    result = (peak * n) // (2 ** ell)
    print_msbs(x, int(result), ell)
    print("peak index =", peak,result)
    print("peak magnitude =", np.abs(W[peak]))
    hj, cj = sort_and_reduce(hj, cj, n)

initial bias = 0.634351183167859

iteration 0
pairs = 10000
bias = 0.634351183167859
filtered pairs = 207
Z size = 256
10101011
10101011
peak index = 209 10973
peak magnitude = 0.3496946854264369

iteration 1
pairs = 9999
bias = 0.42070834604477625
filtered pairs = 9999
Z size = 256
10101011
10110010
peak index = 218 11445
peak magnitude = 17.112470974843927

iteration 2
pairs = 9998
bias = 0.9987297191484086
filtered pairs = 9998
Z size = 256
10101011
10111111
peak index = 233 12233
peak magnitude = 39.01048004642657

iteration 3
pairs = 9997
bias = 0.999695357175812
filtered pairs = 9997
Z size = 256
10101011
1001100
peak index = 93 4882
peak magnitude = 39.04654879595442
